# 04 — Assumption Tests

**What:** A notebook that formally tests the Black-Scholes assumptions against the MXN/USD return series using four statistical tests.

**Why:** Notebook 03 showed the problems visually. This notebook proves them statistically. The distinction matters: a plot is suggestive, a test statistic with a p-value is evidence. Every modeling choice that follows — GARCH over constant volatility, Student-t over normal — gets its formal justification here.

**How:** Four tests, in logical order:
1. **ARCH-LM** — is volatility autocorrelated? (tests the constant volatility assumption)
2. **Jarque-Bera** — are returns normally distributed? (tests the normality assumption)
3. **Ljung-Box** — what is autocorrelated: returns or their squares? (identifies the GARCH signature)
4. **ADF** — are returns stationary? (prerequisite check for GARCH estimation)

**Outline:**

0. Setup
1. ARCH-LM Test — Constant Volatility
2. Jarque-Bera Test — Normality
3. Ljung-Box Test — Autocorrelation Structure
4. ADF Test — Stationarity
5. Summary Table
6. Conclusion

---
---
## 0. Setup

In [14]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.data.describe import (
    compute_returns,
    run_arch_lm,
    run_jarque_bera,
    run_ljung_box,
    run_adf,
    run_all_tests,
    tests_to_frame
)

prices = pd.read_csv(ROOT / 'data' / 'raw' / 'mxn_usd.csv',
                     index_col=0, parse_dates=True).squeeze()
returns = compute_returns(prices)
print(f'Returns: {len(returns):,} observations  '
      f'({returns.index[0].date()} to {returns.index[-1].date()})')


ImportError: cannot import name 'tests_to_frame' from 'src.data.describe' (/Users/Aex/volatility_regimes/src/data/describe.py)

---
---
## 1. ARCH-LM Test — Constant Volatility

The Black-Scholes model assumes volatility $\sigma$ is constant. If that were true, the squared returns $r_t^2$ — which proxy for the magnitude of daily moves — would show no autocorrelation: yesterday's volatility would tell us nothing about today's.

The **ARCH-LM test** (Engle, 1982) formalises this. It regresses $r_t^2$ on its own lags and tests whether those lags have any explanatory power:

$$LM = T \cdot R^2 \sim \chi^2(q)$$

**Null hypothesis $H_0$:** No ARCH effects — volatility is not autocorrelated.

Rejecting $H_0$ means that past squared returns predict future squared returns, which directly violates the constant volatility assumption and justifies a GARCH model.

In [ ]:
arch = run_arch_lm(returns, nlags=10)

print(f'ARCH-LM statistic (LM) : {arch["lm_stat"]:,.4f}')
print(f'F-statistic            : {arch["f_stat"]:,.4f}')
print(f'p-value (LM)           : {arch["lm_pvalue"]:.4e}')
print(f'Decision               : {"Reject H\u2080" if arch["lm_pvalue"] < 0.05 else "Fail to reject H\u2080"}')


ARCH-LM statistic (LM) : 1463.3399
F-statistic            : 187.8864
p-value (LM)           : 2.0855e-308
Decision               : Reject H₀


**What the test found:** The LM statistic is $1{,}463.34$ with $p \approx 2.1 \times 10^{-308}$. The null of no ARCH effects is rejected at every conventional significance level by an enormous margin.

**What it means:** Volatility is strongly autocorrelated. Large moves are followed by large moves, and calm periods are followed by calm periods. This is not a marginal result — the test statistic is over a thousand times the critical value.

**Implication:** A constant-volatility model is not merely imprecise for this series — it is structurally wrong. GARCH(1,1) is the appropriate next step: it models the conditional variance as a function of past squared residuals and past variance, directly capturing this persistence.

---
---
## 2. Jarque-Bera Test — Normality

Black-Scholes assumes log-returns are normally distributed. The **Jarque-Bera test** tests this by measuring how much the empirical skewness and kurtosis deviate from the normal distribution's values (0 and 3 respectively):

$$JB = \frac{n}{6}\left(S^2 + \frac{(K-3)^2}{4}\right) \sim \chi^2(2)$$

**Null hypothesis $H_0$:** Returns are normally distributed ($S = 0$, $K = 3$).

In [ ]:
jb = run_jarque_bera(returns)

print(f'Jarque-Bera statistic  : {jb["jb_stat"]:,.4f}')
print(f'p-value                : {jb["jb_pvalue"]:.4e}')
print(f'Excess kurtosis (K-3)  : {jb["excess_kurtosis"]:.4f}  (normal = 0)')
print(f'Skewness               : {jb["skewness"]:.4f}  (normal = 0)')
print(f'Decision               : {"Reject H\u2080" if jb["jb_pvalue"] < 0.05 else "Fail to reject H\u2080"}')


Jarque-Bera statistic  : 29,801.1331
p-value                : 0.0000e+00
Excess kurtosis (K-3)  : 10.3110  (normal = 0)
Skewness               : 0.7794  (normal = 0)
Decision               : Reject H₀


**What the test found:** The JB statistic is $29{,}801.13$ with $p \approx 0$ (indistinguishable from zero at machine precision). Excess kurtosis is $10.31$ — more than ten times the normal distribution's value of zero. Skewness is $+0.78$, indicating mild right-tail asymmetry.

**What it means:** The return distribution has dramatically heavier tails than a normal distribution predicts. To use the rubber band analogy from notebook 03: extreme snaps happen far more often than normality assumes, and when they do, they are slightly more likely to go in the positive direction (peso appreciations) than the negative. The kurtosis result is what drives the modeling choice — the skewness is a secondary observation.

**Implication:** Using Gaussian innovations in GARCH would systematically underestimate the probability of extreme moves. Student-t innovations with estimated degrees of freedom $\hat{\nu} \approx 4.6$ directly address this: the lower $\nu$, the heavier the tails. This parameter is estimated from the data, not assumed.

**A note on the implied degrees of freedom.**

The Student-t distribution with $\nu$ degrees of freedom has a known 
theoretical excess kurtosis:

$$\text{Excess Kurtosis} = \frac{6}{\nu - 4} \quad \text{for } \nu > 4$$

This means we can invert the formula to get a rough estimate of $\nu$ 
directly from the empirical kurtosis, without fitting any model:

$$\hat{\nu}_{\text{moments}} = \frac{6}{K} + 4$$

Plugging in our empirical excess kurtosis $K = 10.31$:

$$\hat{\nu}_{\text{moments}} = \frac{6}{10.31} + 4 \approx 4.6$$

This is a **moment-matching estimate** or **Method of Moments (MoM)** — we are asking: "which 
Student-t distribution would produce the kurtosis we observed?" 
It is not a maximum likelihood estimate, and it does not use the 
full shape of the return distribution. It should be thought of as a first 
approximation that tells us roughly where to expect the MLE 
estimate to land when we fit GARCH(1,1)-$t$ in notebook 05.

---
---
## 3. Ljung-Box Test — Autocorrelation Structure

The ARCH-LM test confirmed that volatility is autocorrelated. The **Ljung-Box test** lets us examine the autocorrelation structure more carefully by applying it to two series separately:

$$Q(m) = n(n+2)\sum_{k=1}^{m}\frac{\hat{\rho}_k^2}{n-k} \sim \chi^2(m)$$

- Applied to **$r_t$**: tests whether returns themselves are predictable
- Applied to **$r_t^2$**: tests whether the magnitude of returns is predictable

The contrast between the two results is the defining GARCH signature: unpredictable returns, predictable variance.

**Null hypothesis $H_0$:** No autocorrelation at lags 1 through $m$.

> **Note:** Returns are demeaned before the test to avoid spurious autocorrelation from a non-zero mean inflating the squared return series.

In [ ]:
lb = run_ljung_box(returns, lags=10)

print('Ljung-Box Q(10) on returns r_t:')
print(f'  Statistic : {lb["lb_stat_r"]:.4f}')
print(f'  p-value   : {lb["lb_pvalue_r"]:.4e}')
print(f'  Decision  : {"Reject H\u2080" if lb["lb_pvalue_r"] < 0.05 else "Fail to reject H\u2080"}')

print('\nLjung-Box Q(10) on squared returns r_t\u00b2:')
print(f'  Statistic : {lb["lb_stat_r2"]:.4f}')
print(f'  p-value   : {lb["lb_pvalue_r2"]:.4e}')
print(f'  Decision  : {"Reject H\u2080" if lb["lb_pvalue_r2"] < 0.05 else "Fail to reject H\u2080"}')


Ljung-Box Q(10) on returns r_t:
  Statistic : 33.6128
  p-value   : 2.1475e-04
  Decision  : Reject H₀

Ljung-Box Q(10) on squared returns r_t²:
  Statistic : 3663.9727
  p-value   : 0.0000e+00
  Decision  : Reject H₀


**What the test found:**
- $Q(10)$ on $r_t$: statistic $= 33.61$, $p = 0.0002$ — returns show some weak autocorrelation at 10 lags, but the magnitude is modest.
- $Q(10)$ on $r_t^2$: statistic $= 3{,}663.98$, $p \approx 0$ — squared returns are massively autocorrelated. The statistic is more than 100 times larger.

**What it means:** The contrast is the key result. Returns are close to unpredictable — consistent with weak-form market efficiency. But the *magnitude* of returns — how big the move was, regardless of direction — is highly persistent. Knowing that yesterday's move was large tells you that today's move is likely to be large too.

**Implication:** This is the GARCH argument in statistical form. The conditional mean is not forecastable, but the conditional variance is. A model that treats variance as constant is ignoring the only forecastable component of the series.

---
---
## 4. ADF Test — Stationarity

GARCH estimation requires the return series to be **stationary** — meaning its statistical properties (mean, variance) do not systematically drift over time. If returns had a unit root (were non-stationary), the GARCH likelihood would be undefined and the parameter estimates unreliable.

The **Augmented Dickey-Fuller test** tests for a unit root:

$$\Delta r_t = \alpha + \beta t + \gamma r_{t-1} + \sum_{i=1}^{p} \delta_i \Delta r_{t-i} + \epsilon_t$$

**Null hypothesis $H_0$:** The series has a unit root (is non-stationary, $\gamma = 0$).

Note: this test is a **prerequisite check**, not a Black-Scholes violation test. Stationarity of returns is actually consistent with Black-Scholes — we are confirming that our data meets the modeling requirements before proceeding.

In [ ]:
adf = run_adf(returns)

print(f'ADF statistic  : {adf["adf_stat"]:.4f}')
print(f'p-value        : {adf["adf_pvalue"]:.4e}')
print(f'Critical values:')
print(f'   1%  :  {adf["cv_1pct"]:.4f}')
print(f'   5%  :  {adf["cv_5pct"]:.4f}')
print(f'  10%  :  {adf["cv_10pct"]:.4f}')
print(f'Decision       : {"Reject H\u2080 \u2014 series is stationary" if adf["adf_pvalue"] < 0.05 else "Fail to reject H\u2080"}')


ADF statistic  : -26.6768
p-value        : 0.0000e+00
Critical values:
   1%  :  -3.4313
   5%  :  -2.8620
   10%  :  -2.5670
Decision       : Reject H₀ — series is stationary


**What the test found:** The ADF statistic is $-26.68$, far below the $1\%$ critical value of $-3.43$. The p-value is $0.000$. The null of a unit root is decisively rejected.

**What it means:** The log-return series is stationary. Its mean and variance do not trend over time — they fluctuate around stable long-run values. This is expected for returns (as opposed to price levels, which are typically non-stationary).

**Implication:** The prerequisite for GARCH estimation is satisfied. We can proceed to fitting GARCH(1,1)-$t$ and EGARCH(1,1)-$t$ in notebook 05 with confidence that the return series meets the stationarity requirement.

---
---
## 5. Summary Table

All four tests consolidated. The final column maps each result to the modeling decision it motivates.

In [ ]:
results = run_all_tests(returns)

pd.set_option('display.max_colwidth', None)
tests_to_frame(results)


,Test,Null Hypothesis,Statistic,p-value,Decision,Modeling Response
0,ARCH-LM,No ARCH effects,"1,463.34",2.09e-308,Reject H₀ ***,"GARCH(1,1) conditional variance"
1,Jarque-Bera,Returns are normal,"29,801.13",≈ 0,Reject H₀ ***,Student-t innovations (ν̂ ≈ 4.6)
2,Ljung-Box (r²),No autocorrelation in r²,"3,663.97",≈ 0,Reject H₀ ***,"GARCH(1,1) conditional variance"
3,ADF,Unit root present,-26.6768,0.0000,Reject H₀ ***,Prerequisite satisfied — proceed


---
---
## 6. Conclusion

All three Black-Scholes assumptions that can be formally tested on this series have been tested and rejected:

- **Constant volatility** — rejected by ARCH-LM ($LM = 1{,}463.34$, $p \approx 0$)
- **Normal returns** — rejected by Jarque-Bera ($JB = 29{,}801.13$, $p \approx 0$, excess kurtosis $= 10.31$)
- **Unpredictable variance** — confirmed by Ljung-Box contrast ($Q(10)$ on $r_t^2 = 3{,}663.98$ vs $Q(10)$ on $r_t = 33.61$)

The fourth test (ADF) confirms the stationarity prerequisite for the models that follow.

These are not borderline results. Every rejection is by several orders of magnitude. The case for GARCH(1,1) with Student-t innovations is not a modeling preference — it is the conclusion that follows directly from the data. Notebook `05_garch_model.ipynb` builds that model.